# Step 14: Organization-Wide Skill Gap Rollup & Severity Classification

## Overview
This notebook aggregates skill gaps across the organization:
1. Roll up missing skills company-wide to identify top organization-wide skill deficits.
2. Apply severity rules:
   - **HIGH**: Missing in $\ge 100$ employees
   - **MEDIUM**: Missing in $50 - 99$ employees
   - **LOW**: Missing in $< 50$ employees
3. Export organizational skill gap summary to `data/processed/organization_skill_gaps_rollup.csv`.


In [1]:
import pandas as pd
import numpy as np
import os

PROCESSED_DIR = os.path.join("..", "data", "processed")

gap_detail = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_skill_gaps_detail.csv"))
gap_summary = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_skill_gaps_summary.csv"))
attr_df = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_attrition_processed.csv"))

print(f"Total gap instances: {len(gap_detail)}")


Total gap instances: 56271


---
## 1. Organization Skill Rollup & Severity Tiering


In [2]:
# Aggregate missing count per skill across the company
org_skills = gap_detail.groupby('Missing_Skill_Name').agg(
    Employees_Lacking_Count=('EmployeeNumber', 'count'),
    Avg_Importance_Weight=('Importance_Weight', 'mean')
).reset_index()

# Apply Severity Classification Rules
def assign_severity(count):
    if count >= 100:
        return 'HIGH'
    elif count >= 50:
        return 'MEDIUM'
    else:
        return 'LOW'

org_skills['Severity_Tier'] = org_skills['Employees_Lacking_Count'].apply(assign_severity)
org_skills.sort_values(by='Employees_Lacking_Count', ascending=False, inplace=True)

print("=== Organization Skill Gap Severity Distribution ===")
print(org_skills['Severity_Tier'].value_counts())

print("\n=== Top 10 High-Severity Organization Skill Gaps ===")
print(org_skills.head(10).to_string(index=False))

out_path = os.path.join(PROCESSED_DIR, "organization_skill_gaps_rollup.csv")
org_skills.to_csv(out_path, index=False)
print(f"\nSaved Org Skill Rollup: {out_path}")


=== Organization Skill Gap Severity Distribution ===
Severity_Tier
HIGH      228
LOW        89
MEDIUM     65
Name: count, dtype: int64

=== Top 10 High-Severity Organization Skill Gaps ===
       Missing_Skill_Name  Employees_Lacking_Count  Avg_Importance_Weight Severity_Tier
Microsoft Office software                      548               3.000000          HIGH
             SAP software                      537               3.000000          HIGH
          Microsoft Excel                      519               3.000000          HIGH
     Microsoft PowerPoint                      513               3.000000          HIGH
    Reading Comprehension                      467               3.817002          HIGH
     Microsoft SharePoint                      465               3.000000          HIGH
               Monitoring                      464               3.335172          HIGH
          Active Learning                      461               3.396725          HIGH
              Mathe